In [1]:
import torch
import torch.nn as nn
import numpy as np

# ==========================================
# 1. DEFINE A DUMMY NEURAL NETWORK
# ==========================================
class VisionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

# ==========================================
# 2. BIT-PACKING HELPER FUNCTIONS
# ==========================================
def text_to_bits(text: str) -> list[int]:
    """Encodes text to a bit list prefixed by 32 bits of payload length."""
    encoded = text.encode("utf-8")
    payload_len = len(encoded)
    full_data = payload_len.to_bytes(4, byteorder="big") + encoded

    bits = []
    for byte in full_data:
        for shift in range(7, -1, -1):
            bits.append((byte >> shift) & 1)
    return bits

def bits_to_text(bits: list[int]) -> str:
    """Reconstructs text from bits using the 32-bit length header."""
    bytes_arr = bytearray()
    for i in range(0, len(bits), 8):
        chunk = bits[i:i+8]
        if len(chunk) < 8:
            break
        val = 0
        for b in chunk:
            val = (val << 1) | b
        bytes_arr.append(val)

    length = int.from_bytes(bytes_arr[:4], byteorder="big")
    return bytes_arr[4:4 + length].decode("utf-8", errors="replace")

# ==========================================
# 3. STEGANOGRAPHY ENGINE (IEEE-754 LSB)
# ==========================================
def inject_payload(model: nn.Module, secret_msg: str):
    """Embeds payload directly into float32 mantissa bit 0."""
    bits = text_to_bits(secret_msg)
    bits_needed = len(bits)

    # Target the heaviest layer (first linear layer)
    target_param = model.net[0].weight.data
    flat_weights = target_param.cpu().numpy().flatten()

    if len(flat_weights) < bits_needed:
        raise ValueError(f"Weight capacity ({len(flat_weights)} bits) < Payload size ({bits_needed} bits)")

    # View float32 array as raw uint32 memory without data conversion
    u32_view = flat_weights.view(np.uint32)

    # Overwrite the lowest bit of the mantissa (Bit 0)
    for i, bit in enumerate(bits):
        u32_view[i] = (u32_view[i] & ~np.uint32(1)) | np.uint32(bit)

    # Reassign modified tensor back to model layer
    restored_tensor = torch.from_numpy(flat_weights.reshape(target_param.shape))
    model.net[0].weight.data.copy_(restored_tensor)
    print(f"[+] Successfully injected {bits_needed} bits ({len(secret_msg)} chars) into model weights.")

def extract_payload(model: nn.Module) -> str:
    """Reads IEEE-754 mantissa bit 0 to extract embedded secret."""
    target_param = model.net[0].weight.data
    flat_weights = target_param.cpu().numpy().flatten()
    u32_view = flat_weights.view(np.uint32)

    # 1. Read first 32 bits to determine payload length
    len_bits = [int(u32_view[i] & 1) for i in range(32)]
    length_bytes = bytearray()
    for i in range(0, 32, 8):
        byte = 0
        for b in len_bits[i:i+8]:
            byte = (byte << 1) | b
        length_bytes.append(byte)

    payload_len = int.from_bytes(length_bytes, byteorder="big")
    total_bits = (4 + payload_len) * 8

    # 2. Extract remaining bits
    all_bits = [int(u32_view[i] & 1) for i in range(total_bits)]
    return bits_to_text(all_bits)

# ==========================================
# 4. EXECUTION DEMO
# ==========================================
torch.manual_seed(42)
model = VisionClassifier()
dummy_input = torch.randn(1, 128)

# Baseline inference
baseline_output = model(dummy_input)

# Secret message to conceal
secret = "CONFIDENTIAL: AES-256-GCM Master Key: 9f82c3e1b04a8721cd5e9a4f"
print(f"Original Secret : '{secret}'\n")

# Inject payload into weights
inject_payload(model, secret)

# Post-injection inference check
modified_output = model(dummy_input)
max_deviation = (baseline_output - modified_output).abs().max().item()

# Recover secret from model
recovered = extract_payload(model)



Original Secret : 'CONFIDENTIAL: AES-256-GCM Master Key: 9f82c3e1b04a8721cd5e9a4f'

[+] Successfully injected 528 bits (62 chars) into model weights.


In [2]:
print("\n--- RESULTS ---")
print(f"Max Float32 Output Shift : {max_deviation:.10f}")
print(f"Outputs Identical        : {torch.allclose(baseline_output, modified_output, atol=1e-6)}")
print(f"Extracted Secret         : '{recovered}'")
print(f"Payload Integrity Verified: {recovered == secret}")


--- RESULTS ---
Max Float32 Output Shift : 0.0000000112
Outputs Identical        : True
Extracted Secret         : 'CONFIDENTIAL: AES-256-GCM Master Key: 9f82c3e1b04a8721cd5e9a4f'
Payload Integrity Verified: True
